In [1]:
!pip install joblib pandas scipy scikit-learn

In [3]:
 #import neccesary libraries

# save and load trained machine learning models.
import joblib

# handling datasets
import pandas as pd

# Provides scientific computing tools
from scipy.sparse import hstack
import scipy.sparse as sp
import os

# Machine learning library that contains
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score

In [4]:
#Loading data 
X_train_text = joblib.load("shared/X_train_text.pkl")
X_test_text = joblib.load("shared/X_test_text.pkl")

# Load metadata features
X_train_meta = joblib.load("shared/X_train_meta.pkl")
X_test_meta = joblib.load("shared/X_test_meta.pkl")

# Load labels
y_train = joblib.load("shared/y_train.pkl")
y_test = joblib.load("shared/y_test.pkl")

In [14]:
X_train_meta.info()

<class 'pandas.core.frame.DataFrame'>
Index: 661918 entries, 122010 to 121958
Data columns (total 3 columns):
 #   Column         Non-Null Count   Dtype
---  ------         --------------   -----
 0   review_length  661918 non-null  int64
 1   vote           661918 non-null  int64
 2   verified       661918 non-null  int64
dtypes: int64(3)
memory usage: 20.2 MB


## Function for printing results 

In [4]:
def print_results(results):
    print('BEST PARAMS: {}\n'.format(results.best_params_))

    means = results.cv_results_['mean_test_score']
    stds = results.cv_results_['std_test_score']
    for mean, std, params in zip(means, stds, results.cv_results_['params']):
        print('{} (+/-{}) for {}'.format(round(mean, 3), round(std * 2, 3), params))

## Testing different models
Here the GridSearch test models with different amount of trees and different depth of trees. 

In [5]:
rf = RandomForestClassifier()
parameters = {
    'n_estimators': [5,50,100],
    'max_depth': [2,10,20]
}

cv = GridSearchCV(rf,parameters)
cv.fit(X_train_text,y_train)
print_results(cv)

BEST PARAMS: {'max_depth': 20, 'n_estimators': 5}

0.859 (+/-0.0) for {'max_depth': 2, 'n_estimators': 5}
0.859 (+/-0.0) for {'max_depth': 2, 'n_estimators': 50}
0.859 (+/-0.0) for {'max_depth': 2, 'n_estimators': 100}
0.86 (+/-0.0) for {'max_depth': 10, 'n_estimators': 5}
0.859 (+/-0.0) for {'max_depth': 10, 'n_estimators': 50}
0.859 (+/-0.0) for {'max_depth': 10, 'n_estimators': 100}
0.865 (+/-0.002) for {'max_depth': 20, 'n_estimators': 5}
0.862 (+/-0.001) for {'max_depth': 20, 'n_estimators': 50}
0.861 (+/-0.001) for {'max_depth': 20, 'n_estimators': 100}


## Creating the best models for tests 

## Checking the accuracy, precision and recall of the best models with test data. 

In [5]:
rf1 = RandomForestClassifier(n_estimators=100,max_depth=20)
rf1.fit(X_train_text, y_train)
rf2 = RandomForestClassifier(n_estimators=50,max_depth=20)
rf2.fit(X_train_text, y_train)
rf3 = RandomForestClassifier(n_estimators=5,max_depth=20)
rf3.fit(X_train_text, y_train)
rf4 = RandomForestClassifier(n_estimators=25,max_depth=20)
rf4.fit(X_train_text, y_train)

RandomForestClassifier(max_depth=20, n_estimators=25)

In [11]:


for mdl in [rf1,rf2,rf3,rf4]:
    y_pred = mdl.predict(X_test_text)
    accuracy = round(accuracy_score(y_test,y_pred), 3)
    precision = round(precision_score(y_test,y_pred, pos_label="positive"), 3)
    recall = round(recall_score(y_test,y_pred, pos_label="positive"), 3)
    print('MAX DEPTH: {} / # OF EST: {} -- A: {} / P: {} / R: {}'.format(mdl.max_depth,
                                                                         mdl.n_estimators,
                                                                         accuracy,
                                                                         precision,
                                                                         recall))

MAX DEPTH: 20 / # OF EST: 100 -- A: 0.86 / P: 0.86 / R: 1.0
MAX DEPTH: 20 / # OF EST: 50 -- A: 0.86 / P: 0.86 / R: 1.0
MAX DEPTH: 20 / # OF EST: 5 -- A: 0.862 / P: 0.862 / R: 0.999
MAX DEPTH: 20 / # OF EST: 25 -- A: 0.86 / P: 0.86 / R: 1.0


## Again the best classifier is the one with max depth of 20 and number of estimators 5 
Accuracy=0.863, Precision=0.862, Recall=0.99

In [7]:
y_predrf3=rf3.predict(X_test_text)

accuracy = accuracy_score(y_test, y_predrf3)

f1_text = f1_score(y_test, y_predrf3,pos_label="positive")

print("Logistic Regression (TF-IDF only)")
print("Accuracy:", accuracy)
print(classification_report(y_test, y_predrf3))

Logistic Regression (TF-IDF only)
Accuracy: 0.862098138747885
              precision    recall  f1-score   support

    negative       0.86      0.04      0.07     23530
    positive       0.86      1.00      0.93    141950

    accuracy                           0.86    165480
   macro avg       0.86      0.52      0.50    165480
weighted avg       0.86      0.86      0.80    165480



## Plus class weight balanced


In [8]:
rf1v3 = RandomForestClassifier(n_estimators=100,max_depth=20,class_weight="balanced")
rf1v3.fit(X_test_text, y_test)
rf2v3 = RandomForestClassifier(n_estimators=50,max_depth=20,class_weight="balanced")
rf2v3.fit(X_test_text, y_test)
rf3v3 = RandomForestClassifier(n_estimators=5,max_depth=20,class_weight="balanced")
rf3v3.fit(X_test_text, y_test)
rf4v3 = RandomForestClassifier(n_estimators=25,max_depth=20,class_weight="balanced")
rf4v3.fit(X_test_text, y_test)

RandomForestClassifier(class_weight='balanced', max_depth=20, n_estimators=25)

In [9]:


for mdl in [rf1v3,rf2v3,rf3v3,rf4v3]:
    y_pred = mdl.predict(X_test_text)
    accuracy = round(accuracy_score(y_test,y_pred), 3)
    precision = round(precision_score(y_test,y_pred, pos_label="positive"), 3)
    recall = round(recall_score(y_test,y_pred, pos_label="positive"), 3)
    print('MAX DEPTH: {} / # OF EST: {} -- A: {} / P: {} / R: {}'.format(mdl.max_depth,
                                                                         mdl.n_estimators,
                                                                         accuracy,
                                                                         precision,
                                                                         recall))

MAX DEPTH: 20 / # OF EST: 100 -- A: 0.854 / P: 0.958 / R: 0.868
MAX DEPTH: 20 / # OF EST: 50 -- A: 0.85 / P: 0.955 / R: 0.866
MAX DEPTH: 20 / # OF EST: 5 -- A: 0.799 / P: 0.945 / R: 0.813
MAX DEPTH: 20 / # OF EST: 25 -- A: 0.839 / P: 0.955 / R: 0.853


In [10]:
y_predrf1v3=rf1v3.predict(X_test_text)

accuracy = accuracy_score(y_test, y_predrf1v3)

f1_text = f1_score(y_test, y_predrf1v3, average="weighted")

print("Logistic Regression (TF-IDF only)")
print("Accuracy:", accuracy)
print(classification_report(y_test, y_predrf1v3))

Logistic Regression (TF-IDF only)
Accuracy: 0.8539883973894126
              precision    recall  f1-score   support

    negative       0.49      0.77      0.60     23530
    positive       0.96      0.87      0.91    141950

    accuracy                           0.85    165480
   macro avg       0.72      0.82      0.76    165480
weighted avg       0.89      0.85      0.87    165480



In [12]:
y_predrf3v3=rf3v3.predict(X_test_text)

accuracy = accuracy_score(y_test, y_predrf1v3)

f1_text = f1_score(y_test, y_predrf3v3, average="weighted")

print("Logistic Regression (TF-IDF only)")
print("Accuracy:", accuracy)
print(classification_report(y_test, y_predrf3v3))

Logistic Regression (TF-IDF only)
Accuracy: 0.8539883973894126
              precision    recall  f1-score   support

    negative       0.39      0.71      0.50     23530
    positive       0.95      0.81      0.87    141950

    accuracy                           0.80    165480
   macro avg       0.67      0.76      0.69    165480
weighted avg       0.87      0.80      0.82    165480



## Creating data frames to concat with meta data 

In [14]:
X_train_df = pd.DataFrame.sparse.from_spmatrix(X_train_text)
X_test_df = pd.DataFrame.sparse.from_spmatrix(X_test_text)

## Combinig two data frames

In [15]:
X_train_combined = pd.concat([X_train_df.reset_index(drop=True), X_train_meta.reset_index(drop=True)], axis=1)
X_test_combined = pd.concat([X_test_df.reset_index(drop=True), X_test_meta.reset_index(drop=True)], axis=1)

## Checking the best models with meta data 
Not tested due to length of the processing and low accuracy score compared to other models without meta data

In [ ]:
rfv2 = RandomForestClassifier()
parameters = {
    'n_estimators': [5,25,50,100],
    'max_depth': [2,10,20]
}

cvv2 = GridSearchCV(rfv2,parameters)
cvv2.fit(X_train_combined,y_train)
print_results(cv)

## Reference
https://www.kaggle.com/code/onadegibert/sentiment-analysis-with-tfidf-and-random-forest

Claude used for debugging the code